In [ ]:
import numpy as np
import pandas as pd
import datetime as dt
import matplotlib.pyplot as plt
import scipy
import statsmodels.api as sm
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from statsmodels.tsa.stattools import adfuller
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
TENORS = [1, 5, 10, 20, 30]
TENOR_COLS = [f'EU_{t}Y' for t in TENORS]
TRAINING_WINDOW = 504
HOLDOUT_WINDOW = 126
TEST_WINDOW = 252
REGIME_WINDOW = 63
MIN_SIGNAL_CONFIDENCE = 0.8
MAX_POSITION_SIZE = 5.0
STOP_LOSS_THRESHOLD = -0.5

In [ ]:
def load_yield_data() -> pd.DataFrame:

    yield_data = pd.read_csv('data/EU_yield_curves_combined.csv')

    expected_columns = ['EU_1Y', 'EU_5Y', 'EU_10Y', 'EU_20Y', 'EU_30Y']

    if not all(col in yield_data.columns for col in expected_columns):
        raise ValueError('The expected columns do not exist in the dataframe')

    yield_data['DATE'] = pd.to_datetime(yield_data['DATE'])
    yield_data.set_index('DATE', inplace = True)

    return yield_data

yield_data = load_yield_data()

In [ ]:
# Check for missing values
print('Missing values in each column:')
print(yield_data.isna().sum())

In [ ]:
# Check for zero variance
print('Variance of each column:')
print(yield_data.var())

In [ ]:
print('Condition number of the data matrix:')
condition_number = np.linalg.cond(yield_data[['EU_1Y', 'EU_5Y', 'EU_10Y', 'EU_20Y', 'EU_30Y']])
print(round(condition_number, 2))

In [ ]:
# Plot yield curve over time
plt.figure(figsize = (15,8))
for col in yield_data.columns:
    plt.plot(yield_data.index, yield_data[col], label=col)
plt.title('EU Yield Curve Evolution')
plt.xlabel('Date')
plt.ylabel('Yield (%)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('images/yield_curve_evolution.png')
plt.show()

In [ ]:
def calculate_duration(yields, tenors = TENORS) -> pd.DataFrame:

    durations = pd.DataFrame(index = yields.index, columns = [f'DUR_{t}Y' for t in tenors])

    for i, tenor in enumerate(tenors):

        col = f'EUR_{tenor}Y'

        durations[f'DUR_{tenor}Y'] = tenor / (1 + yields[col] / 100)

    return durations

In [ ]:
def calculate_dv01(yields, durations, face_value: int = 100) -> pd.DataFrame:

    dv01 = pd.DataFrame(index = yields.index, columns = [f'DV01_{t}Y' for t in tenors])

    for tenor in tenors:

        bond_price = face_value

        dur_col = f'DUR_{tenor}Y'
        dv01_col = f'DV01_{tenor}Y'

        dv01[dv01_col] = durations[dur_col] / bond_price * 0.0001

    return dv01

In [ ]:
def calculate_carry_and_rolldown(
        yield_df: pd.DataFrame,
        funding_rates: pd.DataFrame = None,
        repo_spread: pd.DataFrame = None,
) -> pd.DataFrame:
    """

    :param yield_df: Yield curve data
    :param funding_rates: Risk-free rates for funding
    :param repo_spreads: Tenor-specific repo spread over funding rates
    :return:
    """

    tenors = TENORS
    carry_rolldown = pd.DataFrame(index = yield_df.index)

    # If funding rates are not provided, approximate using shortest tenor
    if funding_rates is None:
        funding_rates = pd.DataFrame(
            yield_df['EU_1Y'].values,
            index = yield_df.index,
            columns = ['Funding_Rate']
        )

    # If repo spread not provided use reasonable defaults
    if repo_spread is None:
        repo_spread = pd.DataFrame(
            index = yield_df.index,
            columns = [f'Repo_Spread_{t}Y' for t in tenors]
        )

        repo_spread['Repo_Spread_1Y'] = 0.05
        repo_spread['Repo_Spread_5Y'] = 0.10
        repo_spread['Repo_Spread_10Y'] = 0.15
        repo_spread['Repo_Spread_20Y'] = 0.20
        repo_spread['Repo_Spread_30Y'] = 0.25

    for date in yield_df.index:
        curve = yield_df.loc[date].values

        # Calculate financing adjusted carry
        carry = np.zeros_like(curve)
        for i, tenor in enumerate(tenors):
            financing_cost = funding_rates.loc[date, 'Funding_Rate'] + repo_spread.loc[date, f'Repo_Spread_{tenor}Y']
            carry[i] = curve[i] - financing_cost

        # Calculate rolldown using cubic spline
        curve_spline = scipy.interpolate.CubicSpline(tenors, curve)
        rolldown = np.zeros_like(curve)
        for i,tenor in enumerate(tenors):
            if tenor > 1:
                rolldown[i] = curve[i] - curve_spline(tenor - 1)
            else:
                rolldown[i] = 0 # No rolldown for shorter tenor

        carry_rolldown.loc[date, [f'Carry_{t}Y' for t in tenors]] = carry
        carry_rolldown.loc[date, [f'Rolldown_{t}Y' for t in tenors]] = rolldown

    for tenor in tenors:
        carry_rolldown[f'Total_{tenor}Y'] = carry_rolldown[f'Carry_{tenor}Y'] + carry_rolldown[f'Rolldown_{tenor}Y']

    return carry_rolldown

carry_rolldown = calculate_carry_and_rolldown(yield_data)

In [ ]:
def detect_regime(yield_df: pd.DataFrame, window:int = REGIME_WINDOW) -> pd.DataFrame:

    df = pd.DataFrame(index = yield_df.index)

    # Slope measures
    df['2s10s_Slope'] = yield_df['EU_10Y'] - yield_df['EU_1Y']

    # Level
    df['Level'] = yield_df['EU_10Y']

    df['Yield_Volatility'] = yield_df['EU_10Y'].rolling(window = 21).std() * 100

    regimes = pd.DataFrame(index = yield_df.index, columns = ['Regime'])

    bull_steepening = (df['2s10s_Slope'].rolling(window = window).mean() > 0) & (df['EU_10Y'].rolling(window = window).mean().diff(window) < 0)
    bull_flattening = (df['2s10s_Slope'].rolling(window = window).mean() < 0) & (df['EU_10Y'].rolling(window = window).mean().diff(window) < 0)
    bear_steepening = (df['2s10s_Slope'].rolling(window = window).mean() > 0) & (df['EU_10Y'].rolling(window = window).mean().diff(window) > 0)
    bear_flattening = (df['2s10s_Slope'].rolling(window = window).mean() < 0) & (df['EU_10Y'].rolling(window = window).mean().diff(window) > 0)
    # TODO: flattening_twist
    # TODO: steepening_twist

    regimes.loc[bull_steepening, 'Regime'] = 'Bull_Steepening'
    regimes.loc[bull_flattening, 'Regime'] = 'Bull_Flattening'
    regimes.loc[bear_steepening, 'Regime'] = 'Bear_Steepening'
    regimes.loc[bear_flattening, 'Regime'] = 'Bear_Flattening'
    # TODO: flattening_twist
    # TODO: steepening_twist

    regimes['Regime'].fillna('Normal', inplace = True)

    return regimes

In [ ]:
def perform_yield_pca(
        yield_df: pd.DataFrame,
        regime_data: pd.DataFrame = None,
        n_components: int = 3,
        window: int = TRAINING_WINDOW,
):

    pca_results = {}

    if regime_data is not None:

        regimes = regime_data['Regime'].unique()

        for regime in regimes:
            regime_idx = regime_data[regime_data['Regime'] == regime].index
            if len(regime_idx) > window // 2:
                regime_data = yield_df.loc[regime_idx]

                pca_pipeline = Pipeline([
                    ('scaler', StandardScaler()),
                    ('pca', PCA(n_components=n_components)),
                ])

                components = pca_pipeline.fit_transform(yield_df)

                pca_df = pd.DataFrame(components,
                                      index = yield_df.index,
                                      columns = [f'PC{i + 1}' for i in range(n_components)]
                                      )

                pca_model = pca_pipeline.named_steps['pca']
                explained_variance = pca_model.explained_variance_ratio_

                pca_results[regime] = {
                    'pca_df': pca_df,
                    'explained_variance': explained_variance,
                    'pca_model': pca_model,
                    'pca_pipeline': pca_pipeline,
                }

    pca_pipeline = Pipeline([
                    ('scaler', StandardScaler()),
                    ('pca', PCA(n_components=n_components)),
                ])

    components = pca_pipeline.fit_transform(yield_df)

    pca_df = pd.DataFrame(components,
                          index = yield_df.index,
                          columns = [f'PC{i + 1}' for i in range(n_components)]
                          )

    pca_model = pca_pipeline.named_steps['pca']
    explained_variance = pca_model.explained_variance_ratio_

    # Store general model
    pca_results['General'] = {
        'pca_df': pca_df,
        'explained_variance': explained_variance,
        'pca_model': pca_model,
        'pca_pipeline': pca_pipeline,
    }

    return pca_results

In [ ]:
def get_appropriate_pca_model(date, pca_results, regime_data: pd.DataFrame = None):

    if regime_data is not None:
        try:
            current_regime = regime_data.loc[date, 'Regime']
            if current_regime in pca_results:
                return pca_results[current_regime]
        except (KeyError, TypeError):
            pass

    return pca_results['General']

In [ ]:
def reconstruct_yield_curve(
        date,
        yield_data,
        pca_results,
        regime_data: pd.DataFrame = None,
):

    actual_yield = yield_data.loc[date]

    model_info = get_appropriate_pca_model(date, pca_results, regime_data)
    pca_pipeline = model_info['pca_pipeline']

    transformed = pca_pipeline.fit_transform([actual_yield])
    reconstructed = pca_pipeline.inverse_transform(transformed)

    return pd.Series(reconstructed[0], index = yield_data.index)

In [ ]:
def calculate_deviations(
        yield_data,
        pca_results,
        regime_data: pd.DataFrame = None,
        window: int = 252
):

    reconstructed_yields = pd.DataFrame(index = yield_data.index, columns = yield_data.columns)

    for date in yield_data.index:
        try:
            reconstructed_yields.loc[date] = reconstruct_yield_curve(
                date, yield_data, pca_results, regime_data
            )
        except:
            continue

    raw_deviations = yield_data - reconstructed_yields

    tenor_vols = raw_deviations.rolling(window=window).std()

    z_score = raw_deviations / tenor_vols

    return {
        'reconstructed_yield': reconstructed_yields,
        'raw_deviations': raw_deviations,
        'tenor_vols': tenor_vols,
        'z_score': z_score,
    }

In [ ]:
def mean_reversion_tests(deviations, window: int = 126) -> pd.DataFrame:

    results = pd.DataFrame(index = deviations.columns, columns = ['ADF_Statistic', 'p-value', 'Mean_Reversion_Score'])

    for col in deviations.columns:
        series = deviations[col].dropna()
        if len(series) > window:
            adf_result = adfuller(series.values, maxlag = int(np.ceil(np.power(len(series)/100, 0.25))))
            results.loc[col, 'ADF_statistic'] = adf_result[0]
            results.loc[col, 'p-value'] = adf_result[1]
            results.loc[col, 'Mean_Reversion_Score'] = 1 - adf_result[1]

    return results

In [ ]:
def estimate_half_life(
        deviations,
        window: int = 126
) -> pd.DataFrame:
    '''
    Estimate the half life of mean reversion for each tenor.
    :param deviations:
    :param window:
    :return:
    '''
    half_lives = pd.DataFrame(index = deviations.columns, columns = deviations.columns)

    for col in deviations.columns:
        for i in range(window, len(deviations)):
            # Use AR(1) model to estimate mean reversion speed.
            y = deviations[col].iloc[i - window:i]
            y = y.dropna()

            if len(y) > window // 2:
                y_lag = y.shift(1).dropna()
                y = y.iloc[1:] # Align with lagged values

                model = sm.OLS(y, sm.add_constant(y_lag))
                try:
                    result = model.fit()
                    phi = result.params[1]

                    if 0 < phi < 1:
                        half_life = -np.log(2) / np.log(phi)
                        half_lives.loc[deviations.index[i], col] = half_life
                    else:
                        half_lives.loc[deviations.index[i], col] = np.nan
                except:
                    half_lives.loc[deviations.index[i], col] = np.nan

    return half_lives

In [ ]:
def calculate_transaction_costs(
        positions,
        dv01
) -> pd.DataFrame:

    bid_ask_bips = {
        'EU_1Y': 0.5,
        'EU_5Y': 1.0,
        'EU_10Y': 1.5,
        'EU_20Y': 2.5,
        'EU_30Y': 3.0,
    }

    transaction_costs = pd.DataFrame(0, index = positions.index, columns = positions.columns)

    for col in positions.columns:
        position_changes = positions[col].diff().fillna(0).abs()
        dv01_col = f'DV01_{col.split("_")[1]}'
        transaction_costs[col] = position_changes * (bid_ask_bips[col]/10000/2) * dv01[dv01_col]

    return transaction_costs

In [ ]:
def calculate_financing_costs(
        positions,
        yield_data,
        funding_rates: pd.DataFrame = None
) -> pd.DataFrame:

    if funding_rates is None:
        funding_rates = pd.DataFrame(
            yield_data['EU_1Y'].values * 0.9, # Assume funding at 90% of 1Y rate
            index = yield_data.index,
            columns = ['Funding_Rate']
        )

    finacing_spreads = {} # TODO

    finacing_costs = pd.DataFrame(0, index = positions.index, columns = positions.columns)

    for col in positions.columns:
        abs_position = positions[col].abs()

        daily_cost = abs_position * (funding_rates['Funding_Rate'] + finacing_spreads[col]) / 25200

        finacing_costs[col] = daily_cost

    return finacing_costs

In [ ]:
def calculate_position_size(
        deviation,
        confidence,
        half_life,
        carry_rolldown,
        dv01,
        tenor,
        max_position: float = MAX_POSITION_SIZE
):
    ...